In [ ]:
%load_ext autoreload
%autoreload 2
# %matplotlib widget
%pdb off

from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython.display import display, HTML
import pyafn
from pyafn import rho, Cd
import random
from scipy.stats import lognorm
from scipy.stats import norm
from scipy.optimize import curve_fit
from emulationHelpers import readEmulationMI, plot_ventilation_model_fit, plot_empirical_model_error_distribution
import seaborn as sns
from flowEmulationUtils import aggregate_window_series_to_room

#close all figures
plt.close('all')
plt.rcParams['figure.dpi'] = 140
im_scaling = .75
plt.rcParams['figure.figsize'] = [6.4 * im_scaling, 4.8 * im_scaling]
plt.rcParams['font.family'] = 'Times New Roman'

home_dir = "./"
display(home_dir)


In [ ]:
flowStatsMI, roomVentilationMI = readEmulationMI(home_dir=home_dir)

y_var = "flux-Norm"
x_var = "p-noInt_optp0-q_model-Norm"
room_type_order = ["cross", "corner", "dual", "single"]
sigma_q_label = "$\\sigma_{q_n}$ Bulk Fit PN"
sigma_p_label = "$\\sigma_{\\Delta C_p}$ Bulk Fit PN"
model_order = [sigma_q_label, sigma_p_label]
model_tick_labels = {
    sigma_q_label: "$\\sigma_{q_n}$",
    sigma_p_label: "$\\sigma_{\\Delta C_p}$",
}


In [ ]:
# Row 1: baseline q model (categorical hue)
fig, ax, xAdjusted, fittedParams = plot_ventilation_model_fit(
    data=flowStatsMI,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ Bulk Fit PN",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=True,
    show_assymptotes=True,
    add_numeric_colorbar=False,
    return_data=True,
    return_params=True
)

flowStatsMI["q_model-Norm-Adjusted"] = xAdjusted

## ASHRAE Ventilation Rates

In [ ]:
windowASHRAE = pd.read_csv(f"{home_dir}/ashrae_exports/windowASHRAE.csv")
data=flowStatsMI

ashrae_lookup = windowASHRAE.set_index(["windowType", "roomA", "C"])["ventilationRate"]
data["ASHRAE"] = data.apply(
    lambda row: ashrae_lookup.get((row["windowType"], row["roomA"], row["C"]), 999) if not row["slAll"] else np.nan, axis=1
)

plot_df = data[data["slAll"] == False].copy()

fig, axs = plt.subplots(1, 3, figsize=(12, 4.8), dpi=160, sharey=True, layout="constrained")

scatter_specs = [
    ("ASHRAE", "ASHRAE Pressures",  "Modeled Flux Velocity"),
    (x_var, "LES Pressures", "Modeled Flux Velocity"),
    (xAdjusted, "LES Pressures", "Fluctuation-Adjusted Flux Velocity"),
]

for ax, (xcol, title, xlabel) in zip(axs, scatter_specs):
    if type(xcol) == str:
        print(f"{xcol}: {len(plot_df[xcol].dropna())}")
    sns.scatterplot(
        data=plot_df,
        x=xcol,
        y=y_var,
        hue="roomType",
        # style="roomType",
        palette="colorblind",
        hue_order=room_type_order,
        alpha=0.65,
        s=20,
        ax=ax,
        legend=ax is axs[0],
    )

    lim_min = np.nanmin([plot_df[x_var].min(), plot_df[y_var].min()])
    lim_max = np.nanmax([plot_df[x_var].max(), plot_df[y_var].max()])
    ax.plot(
        [lim_min, lim_max],
        [lim_min, lim_max],
        "r--",
        linewidth=1.2,
        alpha=0.7,
        label="1:1" if ax is axs[0] else None,
    )

    ax.set_title(title, fontsize=18)
    ax.set_xlabel(xlabel, fontsize=16)
    ax.set_ylabel("LES Flux Velocity", fontsize=16)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=14)
    # ax.set_xlim(-0.6, 0.6)
    # ax.set_ylim(-0.6, 0.6)


handles, labels = axs[0].get_legend_handles_labels()
for ax in axs:
    if ax.get_legend() is not None:
        ax.get_legend().remove()

# fig.legend(
#     handles,
#     labels,
#     loc="center left",
#     bbox_to_anchor=(0.90, 0.5),
#     fontsize=12,
#     # title="Room A / Window Type",
#     title_fontsize=13,
#     frameon=False,
# )

fig.suptitle("Window-Level Ventilation Comparison", fontsize=20)
fig.subplots_adjust(left=0.08, right=0.86, top=0.83, bottom=0.18, wspace=0.30)

In [ ]:
data_WA_mean = data[data["slAll"] == False].groupby(["windowType", "roomA"])[y_var].mean().reset_index()

windowTypeOrder = data_WA_mean["windowType"].dropna().unique()
plt.figure()
sns.lineplot(data=data_WA_mean, x="roomA", y=y_var, hue="windowType", palette="tab10", hue_order=windowTypeOrder)
sns.lineplot(data=windowASHRAE, x="roomA", y="ventilationRate", hue="windowType", palette="tab10", linestyle='--', hue_order=windowTypeOrder)

In [ ]:
roomVentilationMI_adj = aggregate_window_series_to_room(
    roomVentilationMI, xAdjusted, out_col="xAdjusted_room", mode="abs_half_sum"
)

In [ ]:
roomASHRAE = pd.read_csv(f"{home_dir}/ashrae_exports/roomASHRAE.csv")

ashrae_lookup = roomASHRAE.set_index(["roomType", "AofA", "C"])["ventilationRate"]
roomVentilationMI_adj["ASHRAE"] = roomVentilationMI_adj.apply(
    lambda row: ashrae_lookup.get((row["roomType"], row["AofA"], row["C"]), np.nan) if not row["slAll"] else np.nan, axis=1
)

plt.figure()
sns.scatterplot(data=roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == False], x="ASHRAE", y=y_var, hue="AofA", palette="viridis")

## ASHRAE PN Bulk-Fit Comparison

In [ ]:
def attach_ashrae(frame, ashrae_frame, keys):
    lookup = ashrae_frame.set_index(keys)["ventilationRate"]
    out = frame.copy()
    out["ASHRAE"] = out.apply(
        lambda row: lookup.get(tuple(row[key] for key in keys), np.nan), axis=1
    )
    if "slAll" in out.columns:
        out.loc[out["slAll"], "ASHRAE"] = np.nan
    return out


def signed_group_specs(frame, sign_col="Sdelp"):
    return [
        ("group", "Flow Entering", frame[sign_col] > 0),
        ("group", "Flow Exiting", frame[sign_col] < 0),
    ]


def compute_error_metrics(frame, target_col, pred_col, model_name, *, group_col=None, group_specs=None):
    if group_specs is None:
        group_specs = [("group", group, frame[group_col] == group) for group in frame[group_col].dropna().unique()]

    rows = []
    for scope, group, mask in [("total", "All", pd.Series(True, index=frame.index)), *group_specs]:
        subset = frame.loc[mask, [target_col, pred_col]].dropna()
        if subset.empty:
            continue
        error = subset[pred_col] - subset[target_col]
        rmse = np.sqrt(np.mean(error**2))
        target_std = np.std(subset[target_col])
        rows.append(
            {
                "model_name": model_name,
                "scope": scope,
                "group": group,
                "rmse": rmse,
                "bias": error.mean(),
                "iqr_error": error.quantile(0.75) - error.quantile(0.25),
                "nrmse": rmse / target_std if target_std > 0 else np.nan,
                "n": len(subset),
            }
        )
    return pd.DataFrame(rows)


def fit_signed_prediction(frame, x_col, y_col, model_func, p0, bounds, *, out_col, model_name, sign_col="Sdelp", min_points=6):
    prediction = pd.Series(np.nan, index=frame.index, dtype=float)
    rows = []

    for s, direction in [(1, "Flow Entering"), (-1, "Flow Exiting")]:
        mask = (frame[sign_col] == s) & frame[x_col].notna() & frame[y_col].notna()
        regdf = frame.loc[mask, [x_col, y_col]].abs().sort_values(by=x_col)
        if len(regdf) < min_points:
            continue

        try:
            popt = curve_fit(model_func, regdf[x_col], regdf[y_col], p0=p0, bounds=bounds)[0]
        except Exception as exc:
            print(f"Skipping {model_name}, {direction}: {exc}")
            continue
        prediction.loc[mask] = model_func(frame.loc[mask, x_col].abs().to_numpy(), *popt) * s
        rows.append(
            {
                "fit_source": "ASHRAE-to-LES Fit",
                "model_name": model_name,
                "opening_group": "Window",
                "direction": direction,
                "cd": Cd * popt[0],
                "sigma": popt[1] if len(popt) > 1 else np.nan,
                "n": len(regdf),
                "x_label": x_col,
                "y_label": y_col,
            }
        )

    return prediction.rename(out_col), pd.DataFrame(rows)


def plot_model_scatter(ax, frame, target_col, pred_col, *, hue_col, palette, hue_order, title, x_label="Prediction"):
    sns.scatterplot(
        data=frame,
        x=pred_col,
        y=target_col,
        hue=hue_col,
        palette=palette,
        hue_order=hue_order,
        alpha=0.72,
        s=24,
        ax=ax,
        legend=False,
    )
    vals = pd.concat([frame[target_col], frame[pred_col]]).dropna()
    lo, hi = vals.min(), vals.max()
    ax.plot([lo, hi], [lo, hi], "r--", linewidth=1.2, alpha=0.75)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_title(title, fontsize=15)
    ax.set_xlabel(x_label, fontsize=13)
    ax.set_ylabel("LES Flux Velocity", fontsize=13)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=11)


def plot_error_summary(metrics_df, palette, model_order, model_tick_labels, title):
    metric_rows = [("rmse", "RMSE"), ("bias", "Bias"), ("iqr_error", "Error IQR")]
    fig, axs = plt.subplots(len(metric_rows), 1, figsize=(7.8, 5.8), dpi=160, sharey=True)
    group_data = metrics_df[metrics_df["scope"] == "group"].copy()
    group_order = [group for group in palette if group in group_data["group"].unique()]

    for ax, (metric_col, metric_label) in zip(axs, metric_rows):
        for model_idx, model_name in enumerate(model_order):
            values = []
            for group in group_order:
                row = group_data[(group_data["model_name"] == model_name) & (group_data["group"] == group)]
                if row.empty:
                    continue
                value = row.iloc[0][metric_col]
                values.append(value)
                ax.scatter(value, model_idx, color=palette[group], edgecolor="black", linewidth=0.5, s=38, zorder=3)
            if len(values) > 1:
                ax.plot(values, [model_idx] * len(values), color="0.35", alpha=0.25, linewidth=1.0, zorder=2)
        ax.axvline(0, color="k", linewidth=0.8, alpha=0.7)
        ax.grid(axis="x", alpha=0.25)
        ax.grid(axis="y", alpha=0.35, linestyle="--")
        ax.set_yticks(range(len(model_order)))
        ax.set_yticklabels([model_tick_labels[name] for name in model_order], fontsize=12)
        ax.set_xlabel(metric_label, fontsize=13)
        ax.tick_params(axis="x", labelsize=11)

    handles = [Line2D([0], [0], color=color, marker="o", linestyle="None", markersize=7, label=group) for group, color in palette.items() if group in group_order]
    if handles:
        fig.legend(handles, [h.get_label() for h in handles], loc="lower center", ncol=len(handles), frameon=False, fontsize=10)
    fig.suptitle(title, fontsize=16)
    fig.tight_layout(rect=[0, 0.08, 1, 0.95])
    return fig, axs


In [ ]:
raw_ashrae_label = "ASHRAE Baseline"
ashrae_q_label = "ASHRAE $\\sigma_{q_n}$ Fit"
ashrae_p_label = "ASHRAE $\\sigma_{\\Delta C_p}$ Fit"
fit_bounds = ([0.1, 0], [np.inf, np.inf])

fit_specs = [
    {
        "model_name": ashrae_q_label,
        "tick_label": "ASHRAE $\\sigma_{q_n}$ Fit",
        "pred_col": "ashrae_sigma_q_fit",
        "room_col": "ashrae_sigma_q_fit_room",
        "model_func": pyafn.ventilationReDecomp_q,
        "p0": [1.0, 0.1],
    },
    {
        "model_name": ashrae_p_label,
        "tick_label": "ASHRAE $\\sigma_{\\Delta C_p}$ Fit",
        "pred_col": "ashrae_sigma_p_fit",
        "room_col": "ashrae_sigma_p_fit_room",
        "model_func": pyafn.ventilationReDecomp_p,
        "p0": [1.0, 0.01],
    },
]

window_plot_df = attach_ashrae(flowStatsMI, windowASHRAE, ["windowType", "roomA", "C"])
window_plot_df = window_plot_df[window_plot_df["slAll"] == False].copy()

fit_param_frames = []
for spec in fit_specs:
    window_plot_df[spec["pred_col"]], params_df = fit_signed_prediction(
        window_plot_df,
        "ASHRAE",
        y_var,
        spec["model_func"],
        spec["p0"],
        fit_bounds,
        out_col=spec["pred_col"],
        model_name=spec["model_name"],
    )
    fit_param_frames.append(params_df)

room_plot_df = roomVentilationMI.copy()
for spec in fit_specs:
    room_plot_df = aggregate_window_series_to_room(
        room_plot_df,
        window_plot_df[spec["pred_col"]],
        out_col=spec["room_col"],
        mode="abs_half_sum",
    )
room_plot_df = attach_ashrae(room_plot_df, roomASHRAE, ["roomType", "AofA", "C"])
room_plot_df = room_plot_df[room_plot_df["slAll"] == False].copy()

model_specs_for_metrics = [
    {"model_name": raw_ashrae_label, "tick_label": "ASHRAE Baseline", "window_col": "ASHRAE", "room_col": "ASHRAE", "evaluation": "raw_ashrae"},
    *[{"model_name": spec["model_name"], "tick_label": spec["tick_label"], "window_col": spec["pred_col"], "room_col": spec["room_col"], "evaluation": "ashrae_to_les_fit"} for spec in fit_specs],
]
model_order = [spec["model_name"] for spec in model_specs_for_metrics]
model_tick_labels = {spec["model_name"]: spec["tick_label"] for spec in model_specs_for_metrics}

window_flow_groups = signed_group_specs(window_plot_df)
window_palette = {"Flow Entering": "red", "Flow Exiting": "blue"}
room_palette = {
    room_type: color
    for room_type, color in zip(room_type_order, sns.color_palette("colorblind", n_colors=len(room_type_order)))
    if room_type in room_plot_df["roomType"].dropna().unique()
}

window_error_metrics_df = pd.concat(
    [compute_error_metrics(window_plot_df, y_var, spec["window_col"], spec["model_name"], group_specs=window_flow_groups).assign(dataset="windows", evaluation=spec["evaluation"]) for spec in model_specs_for_metrics],
    ignore_index=True,
)
room_error_metrics_df = pd.concat(
    [compute_error_metrics(room_plot_df, y_var, spec["room_col"], spec["model_name"], group_col="roomType").assign(dataset="rooms", evaluation=spec["evaluation"]) for spec in model_specs_for_metrics],
    ignore_index=True,
)
ashrae_error_metrics_export_df = pd.concat([window_error_metrics_df, room_error_metrics_df], ignore_index=True)[
    ["dataset", "evaluation", "model_name", "scope", "group", "rmse", "bias", "iqr_error", "nrmse", "n"]
]
ashrae_fit_params_export_df = pd.concat(fit_param_frames, ignore_index=True)[
    ["fit_source", "model_name", "opening_group", "direction", "cd", "sigma", "n", "x_label", "y_label"]
]

window_match_summary = pd.DataFrame({"dataset": ["windows"], "rows": [len(window_plot_df)], "ashrae_matches": [window_plot_df["ASHRAE"].notna().sum()]})
room_match_summary = pd.DataFrame({"dataset": ["rooms"], "rows": [len(room_plot_df)], "ashrae_matches": [room_plot_df["ASHRAE"].notna().sum()]})


In [ ]:
plot_specs = [("ASHRAE", raw_ashrae_label), *[(spec["pred_col"], spec["model_name"]) for spec in fit_specs]]
room_plot_specs = [(next(spec["room_col"] for spec in model_specs_for_metrics if spec["model_name"] == label), label) for _, label in plot_specs]
hue_order = [room for room in room_type_order if room in room_palette]

fig_fit, axs_fit = plt.subplots(2, len(plot_specs), figsize=(12.0, 7.0), dpi=160)
for col_idx, ((window_col, label), (room_col, _)) in enumerate(zip(plot_specs, room_plot_specs)):
    plot_model_scatter(
        axs_fit[0, col_idx],
        window_plot_df,
        y_var,
        window_col,
        hue_col="roomType",
        palette=room_palette,
        hue_order=hue_order,
        title="",
        x_label=model_tick_labels[label],
    )
    plot_model_scatter(
        axs_fit[1, col_idx],
        room_plot_df,
        y_var,
        room_col,
        hue_col="roomType",
        palette=room_palette,
        hue_order=hue_order,
        title="",
        x_label=model_tick_labels[label],
    )

axs_fit[0, 0].set_ylabel("Window LES Flux Velocity", fontsize=13)
axs_fit[1, 0].set_ylabel("Room LES Flux Velocity", fontsize=13)
for ax in axs_fit[:, 1:].ravel():
    ax.set_ylabel("")
fig_fit.suptitle("ASHRAE Fits Evaluated Against LES", fontsize=18)
fig_fit.tight_layout(rect=[0, 0, 1, 0.95])

fig_window_error, axs_window_error = plot_error_summary(
    window_error_metrics_df,
    window_palette,
    model_order,
    model_tick_labels,
    "Window ASHRAE Prediction Errors Relative to LES",
)
fig_room_error, axs_room_error = plot_error_summary(
    room_error_metrics_df,
    room_palette,
    model_order,
    model_tick_labels,
    "Room ASHRAE Prediction Errors Relative to LES",
)


In [ ]:
import os

ashrae_export_dir = os.path.join(home_dir, "ashrae_exports")
os.makedirs(ashrae_export_dir, exist_ok=True)

ashrae_export_tables = {
    "ashrae_error_metrics.csv": ashrae_error_metrics_export_df,
    "ashrae_fit_params.csv": ashrae_fit_params_export_df,
}
ashrae_export_paths = {filename: os.path.join(ashrae_export_dir, filename) for filename in ashrae_export_tables}
for filename, df in ashrae_export_tables.items():
    df.to_csv(ashrae_export_paths[filename], index=False)

ashrae_export_manifest_df = pd.DataFrame(
    {
        "table_name": list(ashrae_export_tables.keys()),
        "rows": [len(df) for df in ashrae_export_tables.values()],
        "path": [ashrae_export_paths[filename] for filename in ashrae_export_tables],
    }
)

display(window_match_summary)
display(room_match_summary)
display(ashrae_error_metrics_export_df)
display(ashrae_fit_params_export_df)
display(ashrae_export_manifest_df)
